# VisionAI Inspector — Stage 6: RAG + LLM

**Goal:** let the system explain *why* a detected defect matters and what
to do about it, by retrieving from real inspection documentation instead
of the vision model just outputting a bare label.

**Knowledge base:** 5 documents you wrote for this project (SOP, defect
definitions, QC acceptance criteria, CAPA guide, troubleshooting guide),
already structured with an explicit metadata schema and a 30-question
evaluation set with expected grounding documents — including 9 questions
specifically designed to have **no answer** in the documents, to test that
the system declines rather than guesses.

**Pipeline:** Documents → chunk at the boundaries the documents already
declare (not a fixed token count) → embed locally with **Sentence
Transformers** (`all-MiniLM-L6-v2`) → store in a **Chroma** vector database
with metadata (document_id, defect_type, severity) → retrieve → generate a
grounded answer with **Gemini**.

**Requires one API key as an environment variable**, set before starting
Jupyter: `GEMINI_API_KEY` (create one at Google AI Studio — free tier).
Embeddings run entirely locally and need no API key at all, just a
one-time model download. If the Sentence Transformer model or Gemini API
is unavailable, this notebook falls back to a clearly-tagged offline
substitute (TF-IDF for embeddings, a template stub for generation) so the
pipeline still runs — but real retrieval/generation quality requires both
the real embedding model and a working Gemini key.


## 1. Setup

In [1]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import chromadb

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from rag_documents import load_knowledge_base, parse_evaluation_questions
from rag_embeddings import build_embedder
from rag_generation import get_llm_client, llm_provider_name, generate_answer

KB_DIR = PROJECT_ROOT / "data" / "knowledge_base"
KNOWLEDGE_FILES = [
    "01_bottle_inspection_sop.md",
    "02_defect_definitions_guide.md",
    "03_qc_acceptance_criteria.md",
    "04_capa_guide.md",
    "05_troubleshooting_guide.md",
]

if not (KB_DIR / KNOWLEDGE_FILES[0]).exists():
    raise FileNotFoundError(
        f"Knowledge base documents not found at {KB_DIR}. "
        "This stage needs your own documents (or drafted equivalents) placed there -- "
        "there's no synthetic fallback for this stage, since document content can't be faked "
        "the way a placeholder image can."
    )

print("GEMINI_API_KEY set:", bool(os.environ.get("GEMINI_API_KEY")))

GEMINI_API_KEY set: False


## 2. Parse the knowledge base into chunks

Chunking follows exactly what `06_metadata_schema.md` specifies: the
defect guide chunks at its bold field headers within each defect block
(inheriting `defect_type` and `severity` from that block); the other four
documents chunk at their explicit `**Section: X**` markers. This keeps
every chunk self-contained and tagged with the metadata your vision
pipeline's output (`defect_type`) can directly filter on later.


In [2]:
chunks = load_knowledge_base(KB_DIR, filenames=KNOWLEDGE_FILES)
chunks_df = pd.DataFrame(chunks)

print(f"Total chunks: {len(chunks)}")
print()
print("Chunks per document:")
print(chunks_df["document_id"].value_counts())
print()
print("Chunks per defect_type:")
print(chunks_df["defect_type"].value_counts(dropna=False))
print()
print("Chunks per severity:")
print(chunks_df["severity"].value_counts(dropna=False))
chunks_df[["chunk_id", "document_id", "section", "defect_type", "severity"]].head(10)

Total chunks: 60

Chunks per document:
document_id
DOC-002    25
DOC-001    12
DOC-004     8
DOC-005     8
DOC-003     7
Name: count, dtype: int64

Chunks per defect_type:
defect_type
NaN              34
contamination    10
broken_small      8
broken_large      8
Name: count, dtype: int64

Chunks per severity:
severity
NaN         35
major       16
critical     8
minor        1
Name: count, dtype: int64


,chunk_id,document_id,section,defect_type,severity
0,DOC-001-1-purpose,DOC-001,1. Purpose,NaN,NaN
1,DOC-001-2-scope,DOC-001,2. Scope,NaN,NaN
2,DOC-001-inspection-environment,DOC-001,Inspection Environment,NaN,NaN
3,DOC-001-equipment,DOC-001,Equipment,NaN,NaN
4,DOC-001-inspection-procedure-overview,DOC-001,Inspection Procedure — Overview,NaN,NaN
5,DOC-001-visual-inspection-steps,DOC-001,Visual Inspection Steps,NaN,NaN
6,DOC-001-lighting-and-positioning,DOC-001,Lighting and Positioning,NaN,NaN
7,DOC-001-inspection-frequency,DOC-001,Inspection Frequency,NaN,NaN
8,DOC-001-defect-identification-handoff,DOC-001,Defect Identification Handoff,NaN,NaN
9,DOC-001-acceptance-rejection-process,DOC-001,Acceptance / Rejection Process,NaN,NaN


## 3. Embed every chunk

A local Sentence Transformer (`all-MiniLM-L6-v2`) turns each chunk's text
into a vector that captures its *meaning*, not just its literal words —
this is what lets a query like "small chip at the rim" retrieve the
`broken_small` definition even though it doesn't use that exact term. It
runs entirely on your machine after a one-time model download — no API
key, no per-call cost or network request. If the model can't be loaded
(e.g. no internet for that first download), this falls back to TF-IDF
(literal keyword overlap only) so the rest of the pipeline can still be
exercised end-to-end — but that fallback is not a real demonstration of
semantic retrieval quality (it independently measured ~95% retrieval
accuracy on this project's evaluation set, which speaks more to how
well-structured the source documents are than to TF-IDF itself).


In [3]:
embed_documents, embed_query, using_real_embeddings = build_embedder()
EMBED_TAG = "" if using_real_embeddings else "[TF-IDF FALLBACK, NOT REAL SEMANTIC EMBEDDINGS] "

chunk_texts = chunks_df["text"].tolist()
chunk_vectors = embed_documents(chunk_texts)
print(f"{EMBED_TAG}Embedded {len(chunk_texts)} chunks -> vector dimension: {chunk_vectors.shape[1]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedded 60 chunks -> vector dimension: 384


## 4. Store in a Chroma vector database

Chroma metadata fields must be plain scalars (no `None`), so `defect_type`
and `severity` are stored as `"general"` / `"unspecified"` when the chunk
applies broadly rather than to one specific defect or severity level —
that sentinel is what lets a metadata filter like *"contamination OR
general"* work correctly at query time, matching the OR-with-null logic
your schema document specifies.


In [4]:
client = chromadb.EphemeralClient()  # in-memory for this notebook session; swap for PersistentClient(path=...) to persist across runs
collection = client.get_or_create_collection("bottle_inspection_kb")

metadatas = []
for c in chunks:
    metadatas.append({
        "document_id": c["document_id"],
        "document_type": c["document_type"],
        "section": c["section"],
        "defect_type": c["defect_type"] or "general",
        "severity": c["severity"] or "unspecified",
    })

collection.add(
    ids=[c["chunk_id"] for c in chunks],
    embeddings=chunk_vectors.tolist(),
    metadatas=metadatas,
    documents=chunk_texts,
)
print(f"{EMBED_TAG}Added {collection.count()} chunks to the vector database.")

Added 60 chunks to the vector database.


## 5. Retrieval, with optional defect_type filtering

`retrieve` embeds the query and does a similarity search. When a
`defect_type` is passed (simulating the vision pipeline's classification
output feeding into this step), it filters to chunks tagged with that
defect type OR tagged `"general"` — exactly the combined semantic + metadata
filter your schema doc describes as the intended vision→RAG integration
point.


In [5]:
def retrieve(query: str, top_k: int = 4, defect_type: str | None = None) -> list[dict]:
    query_vector = embed_query(query)

    where = None
    if defect_type is not None:
        where = {"$or": [{"defect_type": defect_type}, {"defect_type": "general"}]}

    results = collection.query(query_embeddings=[query_vector.tolist()], n_results=top_k, where=where)

    retrieved = []
    for chunk_id, doc_text, meta, distance in zip(
        results["ids"][0], results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        retrieved.append({
            "chunk_id": chunk_id, "text": doc_text, "document_id": meta["document_id"],
            "section": meta["section"], "defect_type": meta["defect_type"],
            "severity": meta["severity"], "distance": distance,
        })
    return retrieved

# Quick manual check before the full evaluation run.
example = retrieve("What is contamination?", top_k=3)
for r in example:
    print(f"{EMBED_TAG}[{r['document_id']} / {r['section']}]  distance={r['distance']:.4f}")

[DOC-002 / Contamination — Definition]  distance=0.3368
[DOC-002 / Contamination — Example]  distance=0.7423
[DOC-002 / Contamination — Typical Location]  distance=0.8012


## 6. Generation with Gemini

The system prompt (`src/rag_generation.py`) explicitly instructs Gemini to
answer only from the retrieved excerpts and to decline outright when they
don't contain the answer — this is the actual hallucination-resistance
mechanism, and it's what the evaluation set's 9 unanswerable questions
test directly.


In [6]:
gemini_client = get_llm_client()   # Gemini if GEMINI_API_KEY is set, else local Ollama (see LLM_PROVIDER)
print("LLM provider:", llm_provider_name(gemini_client))
GEN_TAG = "" if gemini_client is not None else "[NO LLM CLIENT, TEMPLATE FALLBACK] "

example_question = "What is contamination?"
example_chunks = retrieve(example_question, top_k=3)
example_result = generate_answer(example_question, example_chunks, client=gemini_client)
print(f"{EMBED_TAG}{GEN_TAG}Q: {example_question}\n")
print(example_result["answer"])

LLM provider: ollama
Q: What is contamination?

Contamination is the presence of foreign material, such as particulate matter, residue, or discoloration, on or within the bottle that should not be present in a normal, defect-free unit.

Source: DOC-001, Inspection Procedure


## 7. Run the full 30-question evaluation set

This is the real test — not just "does it run," but "does it retrieve the
right documents and refuse the right questions." For each question:

- **Retrieval hit**: does at least one of the question's expected
  `document_id`s appear among the top-`k` retrieved chunks?
- **Refusal check** (hallucination questions only): does the generated
  answer contain the required decline phrase, rather than a guessed
  answer?


In [7]:
eval_questions = parse_evaluation_questions(KB_DIR / "07_rag_test_questions.md")
print(f"Loaded {len(eval_questions)} evaluation questions ({sum(q['is_hallucination_test'] for q in eval_questions)} hallucination tests).")

REFUSAL_PHRASE = "does not contain enough information"
TOP_K = 4

import time
PACE_SECONDS = 5 if llm_provider_name(gemini_client) == "gemini" else 0  # stay under free-tier requests-per-minute limits

eval_rows = []
for q in eval_questions:
    retrieved = retrieve(q["question"], top_k=TOP_K)
    retrieved_doc_ids = [r["document_id"] for r in retrieved]

    retrieval_hit = None
    if q["expected_docs"]:
        retrieval_hit = any(doc in retrieved_doc_ids for doc in q["expected_docs"])

    result = generate_answer(q["question"], retrieved, client=gemini_client)
    time.sleep(PACE_SECONDS)
    correctly_refused = REFUSAL_PHRASE.lower() in result["answer"].lower() if q["is_hallucination_test"] else None

    eval_rows.append({
        "question": q["question"][:80],
        "category": q["category"],
        "expected_docs": q["expected_docs"],
        "retrieved_docs": retrieved_doc_ids,
        "retrieval_hit": retrieval_hit,
        "is_hallucination_test": q["is_hallucination_test"],
        "correctly_refused": correctly_refused,
        "answer": result["answer"],
        "used_fallback": result["used_fallback"],
        "fallback_reason": result.get("fallback_reason"),
    })

eval_df = pd.DataFrame(eval_rows)
print(f"\n{EMBED_TAG}{GEN_TAG}Evaluation complete.")

Loaded 30 evaluation questions (9 hallucination tests).

Evaluation complete.


In [8]:
answerable = eval_df[eval_df["expected_docs"].apply(lambda x: bool(x))]
retrieval_accuracy = answerable["retrieval_hit"].mean()
print(f"{EMBED_TAG}Retrieval accuracy (correct document in top-{TOP_K}): {retrieval_accuracy:.1%}  ({answerable['retrieval_hit'].sum()}/{len(answerable)} questions)")

hallucination_rows = eval_df[eval_df["is_hallucination_test"]]
refusal_rate = hallucination_rows["correctly_refused"].mean()
print(f"{GEN_TAG}Correct-refusal rate on unanswerable questions: {refusal_rate:.1%}  ({hallucination_rows['correctly_refused'].sum()}/{len(hallucination_rows)} questions)")

print("\nMisses (expected doc NOT in top-k):")
misses = answerable[answerable["retrieval_hit"] == False]
print(misses[["question", "expected_docs", "retrieved_docs"]] if len(misses) else "None")

print("\nHallucination questions that were NOT correctly refused:")
bad_refusals = hallucination_rows[hallucination_rows["correctly_refused"] == False]
print(bad_refusals[["question", "answer"]] if len(bad_refusals) else "None")

Retrieval accuracy (correct document in top-4): 95.0%  (19/20 questions)
Correct-refusal rate on unanswerable questions: 77.8%  (7/9 questions)

Misses (expected doc NOT in top-k):
                                             question expected_docs  \
16  If a human inspector overrides the automated s...     [DOC-003]   

                          retrieved_docs  
16  [DOC-005, DOC-005, DOC-005, DOC-001]  

Hallucination questions that were NOT correctly refused:
                                             question  \
28  What brand of camera or lighting equipment sho...   
29     What is the shelf life of the bottled product?   

                                               answer  
28  The provided documentation does not specify a ...  
29  The shelf life of the bottled product is not s...  


## 8. A few example question/answer pairs, for a manual sanity read

In [9]:
for _, row in eval_df.iloc[[0, 12, 21, 25]].iterrows():
    print(f"Q ({row['category']}): {row['question']}")
    print(f"A: {row['answer'][:400]}")
    print("-" * 80)

Q (Basic Retrieval): What is contamination?
A: Contamination refers to the presence of foreign material, such as particulate matter, residue, or discoloration, on or within the bottle that should not be present in a normal, defect-free unit.

Source: DOC-002, Contamination — Definition
--------------------------------------------------------------------------------
Q (Reasoning Questions): A bottle has a small visible chip at the rim. What should happen to it?
A: The bottle should be classified as broken_small and rejected.

Source: DOC-002 — Broken (Small) — Example
--------------------------------------------------------------------------------
Q (Hallucination / Unanswerable Questions): What exact temperature should the inspection facility be maintained at?
A: The provided documentation does not contain enough information to answer this.

Source: None
--------------------------------------------------------------------------------
Q (Hallucination / Unanswerable Questions): What ISO

## 9. Save Stage 6 outputs

In [10]:
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
eval_df.to_csv(PROCESSED_ROOT / "rag_evaluation_results.csv", index=False)
chunks_df.to_csv(PROCESSED_ROOT / "rag_chunks.csv", index=False)
print(f"Saved evaluation results and parsed chunk table to {PROCESSED_ROOT}")

if not using_real_embeddings or gemini_client is None:
    print("\nNOTE: local embeddings and/or Gemini API were unavailable during this run.")
    print("Retrieval accuracy / refusal rate above are NOT representative of the real pipeline --")
    print("ensure the Sentence Transformer model can download and GEMINI_API_KEY is set, then re-run.")

Saved evaluation results and parsed chunk table to c:\Users\Vandhana\Downloads\visionai-inspector\visionai-inspector\data\processed


## 10. Summary — Stage 6 complete

What we built:

- A document parser (`src/rag_documents.py`) that chunks at the boundaries
  your documents and metadata schema already declare, verified chunk-by-
  chunk against the real files (60 chunks, counts matching by hand).
- Local Sentence Transformer embeddings (`src/rag_embeddings.py`) — no API
  key, no per-call cost — with a clearly-tagged offline fallback.
- A Chroma vector database with metadata filtering matching your schema's
  vision→RAG integration design.
- Gemini-based generation (`src/rag_generation.py`) with an explicit
  decline-rather-than-guess instruction, and granular error handling that
  distinguishes a missing key, an invalid key, rate limits, and network
  errors rather than crashing or silently mislabeling a fallback as real.
- A full run of your 30-question evaluation set, reporting real retrieval
  accuracy and hallucination-refusal rate — not just "it runs."

**Next (Stage 7):** VLM — reasoning over an image and a question together
("what defect is visible here?"), to complement rather than replace the
specialized vision models built in Stages 2-5.
